#Cell 1 — Install packages

In [ ]:
!pip -q install biopython scikit-learn scipy pandas numpy matplotlib seaborn
!pip -q install fair-esm
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu

print("Installation complete.")

#Cell 2 — Upload the two files

In [ ]:

from google.colab import files
import os

print("Upload BOTH files:")
print("1. final_model.pkl")
print("2. external_validation_peptides.csv")

uploaded = files.upload()

print("\nUploaded files:")
for f in uploaded:
    print(" -", f)

MODEL_PATH = "final_model.pkl"
CSV_PATH = "external_validation_clean.csv"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"{MODEL_PATH} was not uploaded."
    )

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"{CSV_PATH} was not uploaded."
    )

print("\nBoth required files are available.")

#Cell 3 — Imports

In [ ]:
import os
import sys
import math
import json
import pickle
import hashlib
import warnings
from collections import Counter
from dataclasses import dataclass, fields, asdict
from pathlib import Path

import numpy as np
import pandas as pd

from Bio.Align import substitution_matrices
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils.ProtParamData import kd

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc,
    brier_score_loss,
    ConfusionMatrixDisplay,
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("Python:", sys.version)
print("pandas:", pd.__version__)

#Cell 4 — Configuration and lookup tables

In [ ]:

class Config:
    input_csv: str = "main_dataset_with_mhc_scores.csv"
    outdir: str = "ml_report_v5"

    seq_col: str = "Peptide"
    label_col: str = "label"
    id_col: str = "dataset_id"

    mhc_rank_col: str = "MHC_Percentile_Rank"
    mhc_ic50_col: str = "MHC_IC50_nM"
    mhc_class_col: str = "MHC_Class_Used"
    mhc_allele_col: str = "MHC_Allele_Best"

    require_mhc_assay: bool = True
    require_known_class: bool = True

    require_tcell_assay: bool = True
    assay_col: str = "Assay"
    tcell_assay_keyword: str = "T cell assay"

    similarity_threshold: float = 0.80
    conflict_consensus_fraction: float = 0.75
    resolve_conflicts: bool = True

    include_dpc: bool = True
    include_anchor_features: bool = True
    autoc_maxlag: int = 5

    use_mhc_features: bool = True
    mhc_allele_min_count: int = 5

    use_esm_embeddings: bool = True
    esm_model_name: str = "esm2_t12_35M_UR50D"
    esm_repr_layer: int = -1
    esm_batch_size: int = 16
    esm_device: str = ""

    var_threshold: float = 1e-5
    corr_threshold: float = 0.90
    mi_top_k: int = 300

    outer_folds: int = 5
    outer_repeats: int = 3
    inner_folds: int = 3
    use_similarity_cv: str = "auto"
    random_search_iter: int = 35
    threshold_metric: str = "mcc"
    ensemble_min_inner_auc: float = 0.70

    save_explainability_artifacts: bool = True
    shap_max_background: int = 250
    lime_n_explanations: int = 3

    random_seed: int = 42
    n_jobs: int = 4

    def __post_init__(self):
        if self.esm_repr_layer is None or self.esm_repr_layer < 0:
            self.esm_repr_layer = ESM_MODEL_INFO.get(
                self.esm_model_name, 12
            )


STANDARD_AA = "ACDEFGHIKLMNPQRSTVWY"
AA_SET = set(STANDARD_AA)

ALL_DIPEPTIDES = [
    a + b for a in STANDARD_AA for b in STANDARD_AA
]

BLOSUM62 = substitution_matrices.load("BLOSUM62")

ESM_MODEL_INFO = {
    "esm2_t6_8M_UR50D": 6,
    "esm2_t12_35M_UR50D": 12,
    "esm2_t30_150M_UR50D": 30,
    "esm2_t33_650M_UR50D": 33,
}

CTD_GROUPS = {
    "Hydrophobicity": {
        1: set("RKEDQN"),
        2: set("GASTPHY"),
        3: set("CLVIMFW")
    },
    "NormVanDerWaalsVolume": {
        1: set("GASTPDC"),
        2: set("NVEQIL"),
        3: set("MHKFRYW")
    },
    "Polarity": {
        1: set("LIFWCMVY"),
        2: set("PATGS"),
        3: set("HQRKNED")
    },
    "Polarizability": {
        1: set("GASDT"),
        2: set("CPNVEQIL"),
        3: set("KMHFRYW")
    },
    "Charge": {
        1: set("KR"),
        2: set("ANCQGHILMFPSTWYV"),
        3: set("DE")
    },
    "SecondaryStructure": {
        1: set("EALMQKRH"),
        2: set("VIYCWFT"),
        3: set("GNPSD")
    },
    "SolventAccessibility": {
        1: set("ALFCGIVW"),
        2: set("RKQEND"),
        3: set("MSPTHY")
    },
}

AA_POLARITY_GRANTHAM = {
    "A": 8.1, "R": 10.5, "N": 11.6, "D": 13.0,
    "C": 5.5, "Q": 10.5, "E": 12.3, "G": 9.0,
    "H": 10.4, "I": 5.2, "L": 4.9, "K": 11.3,
    "M": 5.7, "F": 5.2, "P": 8.0, "S": 9.2,
    "T": 8.6, "W": 5.4, "Y": 6.2, "V": 5.9
}

AA_VOLUME_ZAMYATNIN = {
    "A": 88.6, "R": 173.4, "N": 114.1, "D": 111.1,
    "C": 108.5, "Q": 143.8, "E": 138.4, "G": 60.1,
    "H": 153.2, "I": 166.7, "L": 166.7, "K": 168.6,
    "M": 162.9, "F": 189.9, "P": 112.7, "S": 89.0,
    "T": 116.1, "W": 227.8, "Y": 193.6, "V": 140.0
}

HYDROPHOBIC_AA = set("AILMFWVC")

print("Configuration loaded.")

#Cell 5 — Feature extraction

In [ ]:
def clean_sequence(seq):
    return str(seq).strip().upper()


def is_valid_peptide(seq):
    seq = clean_sequence(seq)
    return len(seq) > 0 and set(seq).issubset(AA_SET)


def aac_features(seq):
    seq = clean_sequence(seq)
    counts = Counter(seq)
    length = len(seq)

    return {
        f"AAC_{aa}": counts.get(aa, 0) / length * 100
        for aa in STANDARD_AA
    }


def dpc_features(seq):
    seq = clean_sequence(seq)
    denom = max(len(seq) - 1, 1)

    counts = Counter(
        seq[i:i+2]
        for i in range(len(seq) - 1)
    )

    return {
        f"DPC_{dp}": counts.get(dp, 0) / denom * 100
        for dp in ALL_DIPEPTIDES
    }


def _ctd_group_of(aa, groups):
    for group_id, members in groups.items():
        if aa in members:
            return group_id
    return None


def ctd_features(seq):
    seq = clean_sequence(seq)
    length = len(seq)
    feats = {}

    for attr, groups in CTD_GROUPS.items():

        group_seq = [
            _ctd_group_of(aa, groups)
            for aa in seq
        ]

        counts = Counter(group_seq)

        for gid in (1, 2, 3):
            feats[
                f"CTD_{attr}_C{gid}"
            ] = counts.get(gid, 0) / length * 100

        denom = max(length - 1, 1)

        transitions = Counter()

        for i in range(length - 1):
            left = group_seq[i]
            right = group_seq[i + 1]

            if left is None or right is None or left == right:
                continue

            transitions[
                tuple(sorted((left, right)))
            ] += 1

        for pair in [(1, 2), (1, 3), (2, 3)]:
            feats[
                f"CTD_{attr}_T{pair[0]}{pair[1]}"
            ] = transitions.get(pair, 0) / denom * 100

        for gid in (1, 2, 3):

            positions = [
                i + 1
                for i, g in enumerate(group_seq)
                if g == gid
            ]

            if not positions:
                distribution = [0.0] * 5
            else:
                n = len(positions)

                idxs = [
                    1,
                    max(1, math.ceil(0.25 * n)),
                    max(1, math.ceil(0.50 * n)),
                    max(1, math.ceil(0.75 * n)),
                    n,
                ]

                distribution = [
                    positions[idx - 1] / length * 100
                    for idx in idxs
                ]

            for pct, val in zip(
                [0, 25, 50, 75, 100],
                distribution
            ):
                feats[
                    f"CTD_{attr}_D{gid}_{pct}"
                ] = val

    return feats


def physicochemical_features(seq):

    seq = clean_sequence(seq)
    feats = {}

    for name, table in [
        ("Hydrophobicity_KD", kd),
        ("Polarity_Grantham", AA_POLARITY_GRANTHAM),
        ("Volume_Zamyatnin", AA_VOLUME_ZAMYATNIN),
    ]:

        values = np.array(
            [table[aa] for aa in seq],
            dtype=float
        )

        feats[
            f"PHYS_{name}_mean"
        ] = float(values.mean())

        feats[
            f"PHYS_{name}_std"
        ] = float(values.std())

    analysis = ProteinAnalysis(seq)

    feats["PHYS_molecular_weight"] = analysis.molecular_weight()
    feats["PHYS_aromaticity"] = analysis.aromaticity()
    feats["PHYS_instability_index"] = analysis.instability_index()

    try:
        feats["PHYS_isoelectric_point"] = (
            analysis.isoelectric_point()
        )
    except Exception:
        feats["PHYS_isoelectric_point"] = np.nan

    feats["PHYS_gravy"] = analysis.gravy()
    feats["PHYS_charge_at_pH7"] = analysis.charge_at_pH(7.0)

    helix, turn, sheet = (
        analysis.secondary_structure_fraction()
    )

    feats["PHYS_ss_fraction_helix"] = helix
    feats["PHYS_ss_fraction_turn"] = turn
    feats["PHYS_ss_fraction_sheet"] = sheet
    feats["PHYS_length"] = len(seq)

    return feats


def dhkr_features(seq):

    seq = clean_sequence(seq)
    length = len(seq)
    counts = Counter(seq)

    feats = {
        f"DHKR_frac_{aa}":
        counts.get(aa, 0) / length * 100
        for aa in "DHKRE"
    }

    feats["DHKR_charged_fraction"] = (
        sum(counts.get(aa, 0) for aa in "DHKRE")
        / length * 100
    )

    feats["DHKR_net_charge_proxy"] = (
        counts.get("K", 0)
        + counts.get("R", 0)
        + 0.1 * counts.get("H", 0)
        - counts.get("D", 0)
        - counts.get("E", 0)
    ) / length

    return feats


def _moran_autocorrelation(seq, table, maxlag):

    values = np.array(
        [table[aa] for aa in clean_sequence(seq)],
        dtype=float
    )

    length = len(values)
    mean_value = values.mean()

    denom = (
        np.sum((values - mean_value) ** 2)
        / length
    )

    out = []

    for lag in range(1, maxlag + 1):

        if lag >= length or denom == 0:
            out.append(0.0)
            continue

        numerator = (
            np.sum(
                (values[:-lag] - mean_value)
                * (values[lag:] - mean_value)
            )
            / (length - lag)
        )

        out.append(float(numerator / denom))

    return out


def autocorrelation_features(seq, maxlag=5):

    seq = clean_sequence(seq)

    maxlag = max(
        0,
        min(maxlag, len(seq) - 1)
    )

    feats = {}

    for name, table in [
        ("Hydrophobicity_KD", kd),
        ("Polarity_Grantham", AA_POLARITY_GRANTHAM),
        ("Volume_Zamyatnin", AA_VOLUME_ZAMYATNIN),
    ]:

        for lag, value in enumerate(
            _moran_autocorrelation(
                seq, table, maxlag
            ),
            start=1
        ):
            feats[
                f"AUTOC_Moran_{name}_lag{lag}"
            ] = value

    return feats


def blosum62_features(seq):

    seq = clean_sequence(seq)

    acc = {
        aa: 0.0
        for aa in STANDARD_AA
    }

    for residue in seq:
        for aa in STANDARD_AA:
            acc[aa] += BLOSUM62[residue][aa]

    return {
        f"BLOSUM_{aa}":
        acc[aa] / len(seq)
        for aa in STANDARD_AA
    }


def terminal_composition_features(seq):

    seq = clean_sequence(seq)
    feats = {}

    for side, window in [
        ("N", seq[:2]),
        ("C", seq[-2:]),
        ("N3", seq[:3]),
        ("C3", seq[-3:])
    ]:

        counts = Counter(window)
        denom = max(len(window), 1)

        for aa in STANDARD_AA:

            feats[
                f"TERM_{side}_AAC_{aa}"
            ] = (
                counts.get(aa, 0)
                / denom
                * 100
            )

    return feats


def anchor_position_features(seq, mhc_class):

    seq = clean_sequence(seq)
    L = len(seq)

    mhc_class = (
        str(mhc_class).strip().upper()
        if mhc_class is not None
        else ""
    )

    feats = {
        "ANCHOR_P2_hydro": 0.0,
        "ANCHOR_Pomega_hydro": 0.0,
        "ANCHOR_P2_is_hydrophobic": 0.0,
        "ANCHOR_Pomega_is_hydrophobic": 0.0,
        "ANCHOR_classII_a1_hydro": 0.0,
        "ANCHOR_classII_a4_hydro": 0.0,
        "ANCHOR_classII_a6_hydro": 0.0,
        "ANCHOR_classII_a9_hydro": 0.0,
    }

    if L == 0:
        return feats

    if mhc_class == "I":

        p2 = seq[1] if L > 1 else seq[0]
        pomega = seq[-1]

        feats["ANCHOR_P2_hydro"] = float(
            kd.get(p2, 0.0)
        )

        feats["ANCHOR_Pomega_hydro"] = float(
            kd.get(pomega, 0.0)
        )

        feats["ANCHOR_P2_is_hydrophobic"] = float(
            p2 in HYDROPHOBIC_AA
        )

        feats["ANCHOR_Pomega_is_hydrophobic"] = float(
            pomega in HYDROPHOBIC_AA
        )

    elif mhc_class == "II":

        idxs = [
            max(
                0,
                min(
                    L - 1,
                    int(round(f * (L - 1)))
                )
            )
            for f in (0.0, 0.33, 0.55, 0.9)
        ]

        keys = [
            "ANCHOR_classII_a1_hydro",
            "ANCHOR_classII_a4_hydro",
            "ANCHOR_classII_a6_hydro",
            "ANCHOR_classII_a9_hydro",
        ]

        for key, idx in zip(keys, idxs):
            feats[key] = float(
                kd.get(seq[idx], 0.0)
            )

    return feats


def extract_all_features(
    seq,
    cfg,
    mhc_class=None
):

    seq = clean_sequence(seq)
    feats = {}

    feats.update(aac_features(seq))

    if cfg.include_dpc:
        feats.update(dpc_features(seq))

    feats.update(ctd_features(seq))
    feats.update(physicochemical_features(seq))
    feats.update(dhkr_features(seq))
    feats.update(
        autocorrelation_features(
            seq,
            maxlag=cfg.autoc_maxlag
        )
    )
    feats.update(blosum62_features(seq))
    feats.update(
        terminal_composition_features(seq)
    )

    if cfg.include_anchor_features:
        feats.update(
            anchor_position_features(
                seq,
                mhc_class
            )
        )

    return feats

#Cell 6 — MHC and ESM-2 features

In [ ]:
import torch
import esm

print("ESM available:", True)
print("CUDA available:", torch.cuda.is_available())


def extract_mhc_features(df, cfg):

    mhc = pd.DataFrame(index=df.index)

    rank = pd.to_numeric(
        df.get(cfg.mhc_rank_col),
        errors="coerce"
    )

    ic50 = pd.to_numeric(
        df.get(cfg.mhc_ic50_col),
        errors="coerce"
    )

    mhc["MHC_rank_missing"] = (
        rank.isna().astype(float)
    )

    mhc["MHC_ic50_missing"] = (
        ic50.isna().astype(float)
    )

    approx_rank_from_ic50 = np.clip(
        np.log10(ic50 + 1.0) * 2.5,
        0,
        100
    )

    approx_ic50_from_rank = np.clip(
        10 ** (rank / 2.5),
        1,
        100000
    )

    mhc["MHC_rank"] = (
        rank
        .fillna(approx_rank_from_ic50)
        .fillna(100.0)
    )

    mhc["MHC_ic50"] = (
        ic50
        .fillna(approx_ic50_from_rank)
        .fillna(50000.0)
    )

    mhc["MHC_log10_ic50"] = np.log10(
        mhc["MHC_ic50"] + 1.0
    )

    mhc["MHC_sqrt_rank"] = np.sqrt(
        mhc["MHC_rank"]
    )

    mhc["MHC_strong_binder"] = (
        (mhc["MHC_rank"] < 0.5)
        |
        (mhc["MHC_ic50"] < 50.0)
    ).astype(float)

    mhc["MHC_good_binder"] = (
        (
            (mhc["MHC_rank"] >= 0.5)
            &
            (mhc["MHC_rank"] < 2.0)
        )
        |
        (
            (mhc["MHC_ic50"] >= 50.0)
            &
            (mhc["MHC_ic50"] < 500.0)
        )
    ).astype(float)

    mhc["MHC_weak_binder"] = (
        (
            (mhc["MHC_rank"] >= 2.0)
            &
            (mhc["MHC_rank"] < 10.0)
        )
        |
        (
            (mhc["MHC_ic50"] >= 500.0)
            &
            (mhc["MHC_ic50"] < 5000.0)
        )
    ).astype(float)

    mhc["MHC_non_binder"] = (
        (mhc["MHC_rank"] >= 10.0)
        &
        (mhc["MHC_ic50"] >= 5000.0)
    ).astype(float)

    mhc["MHC_binding_score"] = np.clip(
        1.0 / (1.0 + mhc["MHC_rank"]),
        0,
        1
    )

    mhc["MHC_log_ic50_score"] = np.clip(
        1.0
        -
        (
            np.log10(
                mhc["MHC_ic50"] + 1
            ) / 5.0
        ),
        0,
        1
    )

    class_used = (
        df.get(
            cfg.mhc_class_col,
            pd.Series(
                "unknown",
                index=df.index
            )
        )
        .astype(str)
        .str.strip()
        .str.upper()
    )

    mhc["MHC_class_I"] = (
        class_used == "I"
    ).astype(float)

    mhc["MHC_class_II"] = (
        class_used == "II"
    ).astype(float)

    length = df[
        cfg.seq_col
    ].astype(str).str.len()

    mhc["MHC_classI_x_short"] = (
        mhc["MHC_class_I"]
        *
        (
            (length >= 8)
            &
            (length <= 11)
        ).astype(float)
    )

    mhc["MHC_classII_x_long"] = (
        mhc["MHC_class_II"]
        *
        (
            (length >= 13)
            &
            (length <= 25)
        ).astype(float)
    )

    mhc["MHC_compatible_length"] = (
        mhc["MHC_class_I"]
        *
        (
            (length >= 8)
            &
            (length <= 11)
        ).astype(float)
        +
        mhc["MHC_class_II"]
        *
        (
            (length >= 12)
            &
            (length <= 25)
        ).astype(float)
    )

    if cfg.mhc_allele_col in df.columns:

        allele = (
            df[cfg.mhc_allele_col]
            .fillna("unknown")
            .astype(str)
            .str.strip()
        )

        def allele_supertype(a):

            a = str(a).upper()

            for prefix, fam in [
                ("HLA-A", "HLA_A"),
                ("HLA-B", "HLA_B"),
                ("HLA-C", "HLA_C"),
                ("HLA-DR", "HLA_DR"),
                ("HLA-DQ", "HLA_DQ"),
                ("HLA-DP", "HLA_DP"),
            ]:

                if a.startswith(prefix):
                    return fam

            return "Other"

        fam = allele.apply(
            allele_supertype
        )

        fam_counts = fam.value_counts()

        valid_fams = fam_counts[
            fam_counts >= cfg.mhc_allele_min_count
        ].index.tolist()

        for f in valid_fams:

            mhc[
                f"MHC_allele_fam_{f}"
            ] = (
                fam == f
            ).astype(float)

    return mhc.fillna(0.0)


def _esm_cache_path(seqs, cfg):

    signature = (
        f"{cfg.esm_model_name}|"
        f"{cfg.esm_repr_layer}|"
        f"{len(seqs)}|"
        f"{seqs[0]}|"
        f"{seqs[-1]}"
    )

    h = hashlib.md5(
        signature.encode()
    ).hexdigest()[:16]

    return Path(
        f".esm_external_cache_{h}.pkl"
    )


def esm2_embedding_features(seqs, cfg):

    if not cfg.use_esm_embeddings:
        return pd.DataFrame(
            index=pd.RangeIndex(len(seqs))
        )

    if not seqs:
        return pd.DataFrame(
            index=pd.RangeIndex(0)
        )

    cache_path = _esm_cache_path(
        seqs,
        cfg
    )

    if cache_path.exists():

        print(
            f"[ESM-2] Loading cache: "
            f"{cache_path}"
        )

        return pd.read_pickle(
            cache_path
        )

    device = (
        cfg.esm_device.strip()
        if cfg.esm_device
        else (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )
    )

    if not hasattr(
        esm.pretrained,
        cfg.esm_model_name
    ):
        raise ValueError(
            f"Unknown ESM model: "
            f"{cfg.esm_model_name}"
        )

    print(
        f"[ESM-2] Loading "
        f"{cfg.esm_model_name}"
        f" on {device}"
    )

    model, alphabet = getattr(
        esm.pretrained,
        cfg.esm_model_name
    )()

    model = model.eval().to(device)

    batch_converter = (
        alphabet.get_batch_converter()
    )

    records = []
    feature_names = None

    total = len(seqs)

    with torch.no_grad():

        for start in range(
            0,
            total,
            cfg.esm_batch_size
        ):

            batch_seqs = seqs[
                start:start + cfg.esm_batch_size
            ]

            batch = [
                (
                    str(start + i),
                    seq
                )
                for i, seq
                in enumerate(batch_seqs)
            ]

            _, strings, tokens = (
                batch_converter(batch)
            )

            tokens = tokens.to(device)

            output = model(
                tokens,
                repr_layers=[
                    cfg.esm_repr_layer
                ],
                return_contacts=False
            )

            reps = (
                output["representations"]
                [cfg.esm_repr_layer]
                .detach()
                .cpu()
                .numpy()
            )

            for row_idx, seq in enumerate(
                strings
            ):

                vector = reps[
                    row_idx,
                    1:len(seq) + 1
                ].mean(axis=0)

                if feature_names is None:

                    feature_names = [
                        f"ESM2_{i:03d}"
                        for i in range(
                            vector.shape[0]
                        )
                    ]

                records.append(
                    vector.astype(float)
                )

            print(
                f"[ESM-2] "
                f"{min(start + len(batch_seqs), total)}"
                f"/{total}"
            )

    out_df = pd.DataFrame(
        records,
        columns=feature_names
    )

    out_df.to_pickle(
        cache_path
    )

    print(
        f"[ESM-2] Saved cache: "
        f"{cache_path}"
    )

    return out_df

#Cell 7 — Build exact external feature matrix

In [ ]:
def build_feature_matrix(df, cfg):

    clean_list = (
        df[cfg.seq_col]
        .astype(str)
        .map(clean_sequence)
        .tolist()
    )

    if len(clean_list) == 0:
        raise ValueError(
            "No peptides available."
        )

    if (
        cfg.include_anchor_features
        and cfg.mhc_class_col in df.columns
    ):
        mhc_class_list = (
            df[cfg.mhc_class_col]
            .astype(str)
            .tolist()
        )
    else:
        mhc_class_list = [
            None
        ] * len(clean_list)

    records = [
        extract_all_features(
            seq,
            cfg,
            mhc_class=cls
        )
        for seq, cls
        in zip(
            clean_list,
            mhc_class_list
        )
    ]

    feat_df = (
        pd.DataFrame
        .from_records(records)
        .fillna(0.0)
    )

    esm_df = (
        esm2_embedding_features(
            clean_list,
            cfg
        )
        .fillna(0.0)
    )

    feat_df = pd.concat(
        [
            feat_df.reset_index(drop=True),
            esm_df.reset_index(drop=True)
        ],
        axis=1
    )

    if cfg.use_mhc_features:

        mhc_df = (
            extract_mhc_features(
                df,
                cfg
            )
            .reset_index(drop=True)
        )

        feat_df = pd.concat(
            [
                feat_df,
                mhc_df
            ],
            axis=1
        )

        print(
            "[features] MHC features:",
            mhc_df.shape[1]
        )

    lead_cols = {}

    if (
        cfg.id_col
        and cfg.id_col in df.columns
    ):
        lead_cols[
            cfg.id_col
        ] = df[cfg.id_col].values

    lead_cols[
        cfg.label_col
    ] = df[cfg.label_col].values

    lead_cols[
        cfg.seq_col
    ] = df[cfg.seq_col].values

    return pd.concat(
        [
            pd.DataFrame(lead_cols),
            feat_df.reset_index(drop=True)
        ],
        axis=1
    )

#Cell 8 — Recreate the custom model classes

In [ ]:
class VarianceCorrelationFilter(
    BaseEstimator,
    TransformerMixin
):

    def __init__(
        self,
        var_threshold=1e-5,
        corr_threshold=0.90
    ):
        self.var_threshold = var_threshold
        self.corr_threshold = corr_threshold

    def fit(self, X, y=None):

        X_df = pd.DataFrame(X).copy()

        self.feature_names_in_ = (
            X_df.columns
            .astype(str)
            .tolist()
        )

        X_df.columns = (
            self.feature_names_in_
        )

        variances = X_df.var(
            axis=0
        )

        cols_var = (
            variances[
                variances >
                self.var_threshold
            ]
            .index
            .tolist()
        )

        corr = (
            X_df[cols_var]
            .corr()
            .abs()
        )

        upper = corr.where(
            np.triu(
                np.ones(
                    corr.shape,
                    dtype=bool
                ),
                k=1
            )
        )

        to_drop = {
            col
            for col in upper.columns
            if any(
                upper[col]
                > self.corr_threshold
            )
        }

        self.selected_columns_ = [
            c
            for c in cols_var
            if c not in to_drop
        ]

        if not self.selected_columns_:
            self.selected_columns_ = (
                cols_var[:1]
                if cols_var
                else self.feature_names_in_[:1]
            )

        return self

    def transform(self, X):

        X_df = pd.DataFrame(X).copy()

        X_df.columns = (
            self.feature_names_in_
        )

        return X_df[
            self.selected_columns_
        ]


class MutualInfoTopKSelector(
    BaseEstimator,
    TransformerMixin
):

    def __init__(
        self,
        k=250,
        random_state=42,
        force_include_prefix="MHC_"
    ):
        self.k = k
        self.random_state = random_state
        self.force_include_prefix = (
            force_include_prefix
        )

    def fit(self, X, y):

        from sklearn.feature_selection import (
            mutual_info_classif
        )

        X_df = pd.DataFrame(X).copy()

        self.feature_names_in_ = (
            X_df.columns
            .astype(str)
            .tolist()
        )

        X_df.columns = (
            self.feature_names_in_
        )

        forced = [
            c
            for c in self.feature_names_in_
            if c.startswith(
                self.force_include_prefix
            )
        ]

        remaining = [
            c
            for c in self.feature_names_in_
            if c not in forced
        ]

        if (
            remaining
            and self.k
            and self.k > 0
        ):

            n_from_mi = max(
                0,
                self.k - len(forced)
            )

            if (
                0 < n_from_mi
                < len(remaining)
            ):

                mi = mutual_info_classif(
                    X_df[remaining],
                    y,
                    random_state=self.random_state
                )

                order = np.argsort(
                    mi
                )[::-1][:n_from_mi]

                selected_from_mi = [
                    remaining[i]
                    for i in order
                ]

            else:
                selected_from_mi = remaining

        else:
            selected_from_mi = remaining

        self.selected_columns_ = (
            forced
            + selected_from_mi
        )

        if not self.selected_columns_:
            self.selected_columns_ = (
                self.feature_names_in_
            )

        return self

    def transform(self, X):

        X_df = pd.DataFrame(X).copy()

        X_df.columns = (
            self.feature_names_in_
        )

        return X_df[
            self.selected_columns_
        ]


class WeightedSoftVotingEnsemble(
    BaseEstimator,
    ClassifierMixin
):

    def __init__(
        self,
        estimators,
        weights,
        threshold=0.5
    ):
        self.estimators = estimators
        self.weights = weights
        self.threshold = threshold
        self.classes_ = np.array([0, 1])

    def fit(self, X, y=None):
        return self

    def predict_proba(self, X):

        probs = np.array([
            est.predict_proba(X)[:, 1]
            for est
            in self.estimators.values()
        ])

        pos = np.average(
            probs,
            axis=0,
            weights=self.weights
        )

        return np.column_stack(
            [1.0 - pos, pos]
        )

    def predict(self, X):

        return (
            self.predict_proba(X)[:, 1]
            >= self.threshold
        ).astype(int)

#Cell 9 — Load and audit the model

In [ ]:
with open(
    MODEL_PATH,
    "rb"
) as f:

    artifact = pickle.load(f)


if not isinstance(
    artifact,
    dict
):
    raise ValueError(
        "final_model.pkl is not a dictionary artifact."
    )


required_keys = [
    "model",
    "feature_columns",
    "config"
]

missing_keys = [
    k
    for k in required_keys
    if k not in artifact
]

if missing_keys:
    raise ValueError(
        f"Model artifact is missing: "
        f"{missing_keys}"
    )


model = artifact["model"]
feature_columns = artifact[
    "feature_columns"
]

raw_cfg = artifact["config"]

valid_fields = {
    f.name
    for f in fields(Config)
}

filtered_cfg = {
    k: v
    for k, v in raw_cfg.items()
    if k in valid_fields
}

cfg = Config(
    **filtered_cfg
)

print("=" * 70)
print("MODEL AUDIT")
print("=" * 70)

print(
    "Model class:",
    type(model).__name__
)

print(
    "Training features:",
    len(feature_columns)
)

print(
    "ESM enabled:",
    cfg.use_esm_embeddings
)

print(
    "ESM model:",
    cfg.esm_model_name
)

print(
    "ESM layer:",
    cfg.esm_repr_layer
)

print(
    "DPC:",
    cfg.include_dpc
)

print(
    "Anchor features:",
    cfg.include_anchor_features
)

print(
    "MHC features:",
    cfg.use_mhc_features
)

print(
    "Training ensemble members:"
)

if "member_inner_cv_auc" in artifact:
    for name, auc_value in (
        artifact[
            "member_inner_cv_auc"
        ].items()
    ):
        print(
            f"  {name}: "
            f"{auc_value:.4f}"
        )

print(
    "Stored threshold:",
    getattr(
        model,
        "threshold",
        0.5
    )
)

print("=" * 70)

#Cell 10 — Load and audit external dataset

In [ ]:
df = pd.read_csv(
    CSV_PATH
)

print(
    f"External dataset shape: "
    f"{df.shape}"
)

print("\nColumns:")
print(
    df.columns.tolist()
)

# Define the full set of expected columns for audit and downstream processing.
expected_columns = [
    "dataset_id",
    "Peptide",
    "Organism",
    "Assay",
    "label",
    "seq_length",
    "MHC_Class_Used",
    "MHC_Percentile_Rank",
    "MHC_IC50_nM",
    "MHC_Allele_Best",
    "Class_I",
    "Class_II",
    "query_organism",
    "max_identity_to_train",
]


for col_to_check in ["query_organism", "max_identity_to_train"]:
    if col_to_check not in df.columns:
        print(f"\nINFO: Column '{col_to_check}' not found in input CSV. Adding with default values.")
        if col_to_check == "max_identity_to_train":
            df[col_to_check] = np.nan
        elif col_to_check == "query_organism":
            df[col_to_check] = "UNKNOWN_ORGANISM"


missing = [
    c
    for c in expected_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"External CSV is missing truly critical expected columns: {missing}"
    )

print("\nLabel distribution:")
print(
    df["label"]
    .value_counts(dropna=False)
)

print("\nMHC class:")
print(
    df["MHC_Class_Used"]
    .value_counts(dropna=False)
)

print("\nDuplicate peptide sequences:",
      df["Peptide"].duplicated().sum())

print("\nMissing values:")
print(
    df[expected_columns]
    .isna()
    .sum()
)


#Cell 11 — Clean sequences

In [ ]:
df["Peptide_clean"] = (
    df["Peptide"]
    .astype(str)
    .map(clean_sequence)
)

valid_mask = (
    df["Peptide_clean"]
    .map(is_valid_peptide)
)

n_invalid = (
    ~valid_mask
).sum()

print(
    "Invalid peptide sequences:",
    int(n_invalid)
)

if n_invalid > 0:

    print(
        "\nInvalid sequences:"
    )

    print(
        df.loc[
            ~valid_mask,
            ["dataset_id", "Peptide"]
        ].head(20)
    )

    df = df.loc[
        valid_mask
    ].copy()

df = df.reset_index(
    drop=True
)

df["Peptide"] = df[
    "Peptide_clean"
]

df.drop(
    columns=["Peptide_clean"],
    inplace=True
)

print(
    "\nPeptides retained:",
    len(df)
)

print(
    "Positive:",
    int(
        (df["label"] == 1).sum()
    )
)

print(
    "Negative:",
    int(
        (df["label"] == 0).sum()
    )
)

Invalid peptide sequences: 0

Peptides retained: 1163
Positive: 476
Negative: 687


#Cell 11b — Match the training population definition (T-cell assay restriction)




In [ ]:
print("=" * 70)
print("ASSAY COMPOSITION OF EXTERNAL SET (before any restriction)")
print("=" * 70)

if cfg.assay_col in df.columns:
    print(df[cfg.assay_col].value_counts(dropna=False))
else:
    print(f"WARNING: '{cfg.assay_col}' column not found -- cannot verify assay composition.")


def restrict_to_assay_supported(df, cfg):
    n_before = len(df)
    keep = pd.Series(True, index=df.index)
    reasons = {}

    if cfg.require_mhc_assay:
        rank = pd.to_numeric(df.get(cfg.mhc_rank_col), errors="coerce")
        ic50 = pd.to_numeric(df.get(cfg.mhc_ic50_col), errors="coerce")
        has_assay = rank.notna() | ic50.notna()
        reasons["dropped_no_assay_value"] = int((~has_assay).sum())
        keep &= has_assay

    if cfg.require_known_class:
        class_used = df.get(cfg.mhc_class_col, pd.Series(index=df.index, dtype=object))
        class_used = class_used.astype(str).str.strip().str.upper()
        has_class = class_used.isin(["I", "II"])
        reasons["dropped_unknown_class"] = int((~has_class & keep).sum())
        keep &= has_class

    out = df.loc[keep].reset_index(drop=True)
    report = {"n_before": n_before, "n_after": int(len(out)), **reasons}
    print(f"[restrict] {n_before} -> {len(out)} peptides after MHC assay-support restriction: {reasons}")
    return out, report


def restrict_to_tcell_assay(df, cfg):
    if not cfg.require_tcell_assay or cfg.assay_col not in df.columns:
        print("[restrict] T-cell-assay filter skipped (require_tcell_assay=False or column missing).")
        return df, {"require_tcell_assay": False}

    n_before = len(df)
    has_tcell = df[cfg.assay_col].astype(str).str.contains(
        cfg.tcell_assay_keyword, na=False, case=False
    )
    out = df.loc[has_tcell].reset_index(drop=True)
    report = {
        "n_before_tcell_filter": n_before,
        "n_after_tcell_filter": int(len(out)),
        "n_dropped_no_tcell_evidence": int((~has_tcell).sum()),
    }
    print(f"[restrict] T-cell-assay filter: {n_before} -> {len(out)} peptides "
          f"({report['n_dropped_no_tcell_evidence']} dropped -- no direct T-cell assay evidence).")
    return out, report


df, _assay_report = restrict_to_assay_supported(df, cfg)
df, _tcell_report = restrict_to_tcell_assay(df, cfg)
df = df.reset_index(drop=True)

print("\nPopulation after matching the training definition:")
print("Total:", len(df))
print("Positive:", int((df[cfg.label_col] == 1).sum()))
print("Negative:", int((df[cfg.label_col] == 0).sum()))

if len(df) == 0:
    raise RuntimeError(
        "No external peptides remain after applying the training notebook's "
        "T-cell-assay restriction. Check the 'Assay' column's actual values "
        "against cfg.tcell_assay_keyword (currently: "
        f"'{cfg.tcell_assay_keyword}')."
    )


#Cell 12 — Extract features

In [ ]:
print("=" * 70)
print("STARTING EXTERNAL FEATURE EXTRACTION")
print("=" * 70)

feat_df = build_feature_matrix(
    df,
    cfg
)

print(
    "\nGenerated feature matrix:",
    feat_df.shape
)

feature_only_cols = [
    c
    for c in feat_df.columns
    if c not in [
        cfg.id_col,
        cfg.label_col,
        cfg.seq_col
    ]
]

print(
    "Generated feature count:",
    len(feature_only_cols)
)

print(
    "Training feature count:",
    len(feature_columns)
)

#Cell 13 — Strict feature compatibility check

In [ ]:
X_raw = feat_df.drop(
    columns=[
        c
        for c in [
            cfg.id_col,
            cfg.label_col,
            cfg.seq_col
        ]
        if c in feat_df.columns
    ]
)

missing_features = [
    c
    for c in feature_columns
    if c not in X_raw.columns
]

extra_features = [
    c
    for c in X_raw.columns
    if c not in feature_columns
]

print(
    "Missing training features:",
    len(missing_features)
)

print(
    "Extra generated features:",
    len(extra_features)
)

if missing_features:

    print(
        "\nFirst missing features:"
    )

    print(
        missing_features[:30]
    )

# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------

allowed_missing = [
    c
    for c in missing_features
    if c.startswith(
        "MHC_allele_fam_"
    )
]

unexpected_missing = [
    c
    for c in missing_features
    if c not in allowed_missing
]

if unexpected_missing:

    raise RuntimeError(
        "\nUnexpected training features "
        "are missing from the external "
        "feature matrix.\n"
        "This is a train/external feature "
        "mismatch and should NOT be silently "
        "zero-filled.\n\n"
        f"Missing features:\n"
        f"{unexpected_missing[:50]}"
    )


for col in allowed_missing:
    X_raw[col] = 0.0

X = X_raw[
    feature_columns
].copy()

print(
    "\nFinal model input:",
    X.shape
)

if list(X.columns) != list(
    feature_columns
):
    raise RuntimeError(
        "Feature order does not match "
        "training feature order."
    )

print(
    "Feature order verified."
)

#Cell 14 — Prediction

In [ ]:
print(
    "Running external predictions..."
)

proba = model.predict_proba(
    X
)[:, 1]

threshold = float(
    getattr(
        model,
        "threshold",
        0.5
    )
)

pred = (
    proba >= threshold
).astype(int)

print(
    "Prediction complete."
)

print(
    f"Decision threshold = "
    f"{threshold:.6f}"
)

print(
    "\nPredicted class distribution:"
)

print(
    pd.Series(
        pred
    ).value_counts()
)

#Cell 15 — External metrics

In [ ]:
y_true = (
    df["label"]
    .astype(int)
    .to_numpy()
)

tn, fp, fn, tp = confusion_matrix(
    y_true,
    pred,
    labels=[0, 1]
).ravel()

sensitivity = (
    tp / (tp + fn)
    if (tp + fn) > 0
    else np.nan
)

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else np.nan
)

roc_auc = (
    roc_auc_score(
        y_true,
        proba
    )
    if len(np.unique(y_true)) == 2
    else np.nan
)

pr_auc = (
    average_precision_score(
        y_true,
        proba
    )
    if (y_true == 1).any()
    else np.nan
)

metrics = {
    "ROC_AUC": roc_auc,
    "PR_AUC": pr_auc,
    "Accuracy": accuracy_score(
        y_true,
        pred
    ),
    "Balanced_Accuracy":
        balanced_accuracy_score(
            y_true,
            pred
        ),
    "Precision":
        precision_score(
            y_true,
            pred,
            zero_division=0
        ),
    "Sensitivity_Recall":
        sensitivity,
    "Specificity":
        specificity,
    "F1":
        f1_score(
            y_true,
            pred,
            zero_division=0
        ),
    "MCC":
        matthews_corrcoef(
            y_true,
            pred
        ),
    "Brier_Score":
        brier_score_loss(
            y_true,
            proba
        ),
    "Threshold":
        threshold,
    "N":
        len(y_true),
    "N_Positive":
        int((y_true == 1).sum()),
    "N_Negative":
        int((y_true == 0).sum()),
    "TN":
        int(tn),
    "FP":
        int(fp),
    "FN":
        int(fn),
    "TP":
        int(tp),
}

metrics_df = pd.DataFrame(
    [metrics]
).T

metrics_df.columns = [
    "Value"
]

display(
    metrics_df
)

#Cell 16 — Bootstrap 95% CI

In [ ]:
def bootstrap_external_ci(
    y_true,
    proba,
    pred,
    n_bootstrap=2000,
    seed=42
):

    rng = np.random.RandomState(
        seed
    )

    n = len(y_true)

    results = {
        "ROC_AUC": [],
        "PR_AUC": [],
        "Accuracy": [],
        "Balanced_Accuracy": [],
        "Precision": [],
        "Sensitivity": [],
        "Specificity": [],
        "F1": [],
        "MCC": [],
        "Brier_Score": [],
    }

    successful = 0

    for _ in range(
        n_bootstrap
    ):

        idx = rng.randint(
            0,
            n,
            n
        )

        yt = y_true[idx]
        pb = proba[idx]
        pp = pred[idx]

        if len(
            np.unique(yt)
        ) < 2:
            continue

        tn, fp, fn, tp = (
            confusion_matrix(
                yt,
                pp,
                labels=[0, 1]
            ).ravel()
        )

        sens = (
            tp / (tp + fn)
            if tp + fn
            else np.nan
        )

        spec = (
            tn / (tn + fp)
            if tn + fp
            else np.nan
        )

        results["ROC_AUC"].append(
            roc_auc_score(
                yt,
                pb
            )
        )

        results["PR_AUC"].append(
            average_precision_score(
                yt,
                pb
            )
        )

        results["Accuracy"].append(
            accuracy_score(
                yt,
                pp
            )
        )

        results[
            "Balanced_Accuracy"
        ].append(
            balanced_accuracy_score(
                yt,
                pp
            )
        )

        results["Precision"].append(
            precision_score(
                yt,
                pp,
                zero_division=0
            )
        )

        results["Sensitivity"].append(
            sens
        )

        results["Specificity"].append(
            spec
        )

        results["F1"].append(
            f1_score(
                yt,
                pp,
                zero_division=0
            )
        )

        results["MCC"].append(
            matthews_corrcoef(
                yt,
                pp
            )
        )

        results["Brier_Score"].append(
            brier_score_loss(
                yt,
                pb
            )
        )

        successful += 1

    rows = []

    for metric_name, values in (
        results.items()
    ):

        values = np.asarray(
            values,
            dtype=float
        )

        values = values[
            np.isfinite(values)
        ]

        if len(values) == 0:
            continue

        rows.append({
            "Metric": metric_name,
            "Estimate": float(
                np.mean(values)
            ),
            "CI_95_Lower": float(
                np.percentile(
                    values,
                    2.5
                )
            ),
            "CI_95_Upper": float(
                np.percentile(
                    values,
                    97.5
                )
            ),
        })

    return (
        pd.DataFrame(rows),
        successful
    )


bootstrap_df, n_success = (
    bootstrap_external_ci(
        y_true,
        proba,
        pred,
        n_bootstrap=2000,
        seed=42
    )
)

print(
    f"Successful bootstrap replicates: "
    f"{n_success}/2000"
)

display(
    bootstrap_df
)

#Cell 17 — ROC, PR and confusion matrix

In [ ]:
os.makedirs(
    "external_validation_results",
    exist_ok=True
)

outdir = Path(
    "external_validation_results"
)

# ---------------- ROC ----------------

fpr, tpr, _ = roc_curve(
    y_true,
    proba
)

roc_auc_plot = auc(
    fpr,
    tpr
)

plt.figure(
    figsize=(7, 6)
)

plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"External validation AUC = {roc_auc_plot:.3f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Random classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve — External Validation"
)

plt.legend(
    loc="lower right"
)

plt.tight_layout()

plt.savefig(
    outdir / "external_ROC_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ---------------- PR ----------------

precision, recall, _ = (
    precision_recall_curve(
        y_true,
        proba
    )
)

ap = average_precision_score(
    y_true,
    proba
)

prevalence = y_true.mean()

plt.figure(
    figsize=(7, 6)
)

plt.plot(
    recall,
    precision,
    linewidth=2,
    label=f"Average Precision = {ap:.3f}"
)

plt.axhline(
    prevalence,
    linestyle="--",
    linewidth=1.5,
    label=f"Prevalence = {prevalence:.3f}"
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Precision–Recall Curve — External Validation"
)

plt.legend(
    loc="lower left"
)

plt.tight_layout()

plt.savefig(
    outdir / "external_PR_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ---------------- Confusion Matrix ----------------

fig, ax = plt.subplots(
    figsize=(6, 6)
)

ConfusionMatrixDisplay.from_predictions(
    y_true,
    pred,
    display_labels=[
        "Non-immunogenic",
        "Immunogenic"
    ],
    ax=ax,
    colorbar=False
)

ax.set_title(
    "Confusion Matrix — External Validation"
)

plt.tight_layout()

plt.savefig(
    outdir / "external_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

#Cell 18 — Save per-peptide predictions

In [ ]:
prediction_cols = [
    c
    for c in [
        "dataset_id",
        "Peptide",
        "Organism",
        "Assay",
        "label",
        "seq_length",
        "MHC_Class_Used",
        "MHC_Percentile_Rank",
        "MHC_IC50_nM",
        "MHC_Allele_Best",
        "Class_I",
        "Class_II",
        "query_organism",
        "max_identity_to_train",
    ]
    if c in df.columns
]

predictions_df = df[
    prediction_cols
].copy()

predictions_df[
    "predicted_probability"
] = proba

predictions_df[
    "predicted_label"
] = pred

predictions_df[
    "correct"
] = (
    predictions_df[
        "label"
    ].astype(int)
    ==
    predictions_df[
        "predicted_label"
    ]
)

prediction_path = (
    outdir
    /
    "external_validation_predictions.csv"
)

predictions_df.to_csv(
    prediction_path,
    index=False
)

print(
    f"Saved: {prediction_path}"
)

display(
    predictions_df.head(10)
)

#Cell 19 — Very important: validation by similarity to training set

In [ ]:
df_eval = predictions_df.copy()

df_eval[
    "max_identity_to_train"
] = pd.to_numeric(
    df_eval[
        "max_identity_to_train"
    ],
    errors="coerce"
)

bins = [
    -np.inf,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    0.95,
    np.inf
]

labels = [
    "<=50%",
    "50–60%",
    "60–70%",
    "70–80%",
    "80–90%",
    "90–95%",
    ">95%"
]

df_eval[
    "identity_group"
] = pd.cut(
    df_eval[
        "max_identity_to_train"
    ],
    bins=bins,
    labels=labels
)

rows = []

for group, g in (
    df_eval
    .groupby(
        "identity_group",
        observed=False
    )
):

    if len(g) < 10:
        continue

    if g["label"].nunique() < 2:
        continue

    rows.append({
        "identity_group": str(group),
        "n": len(g),
        "positive": int(
            (g["label"] == 1).sum()
        ),
        "negative": int(
            (g["label"] == 0).sum()
        ),
        "ROC_AUC": roc_auc_score(
            g["label"],
            g["predicted_probability"]
        ),
        "PR_AUC": average_precision_score(
            g["label"],
            g["predicted_probability"]
        ),
    })

similarity_results = pd.DataFrame(
    rows
)

display(
    similarity_results
)

similarity_results.to_csv(
    outdir
    /
    "performance_by_training_similarity.csv",
    index=False
)

#Cell 20 — MHC class and organism subgroup validation

In [ ]:
def subgroup_auc_table(
    df,
    group_col,
    min_n=10
):

    rows = []

    for value, g in (
        df.groupby(
            group_col,
            dropna=False
        )
    ):

        if len(g) < min_n:
            continue

        if g["label"].nunique() < 2:
            continue

        rows.append({
            "group_variable": group_col,
            "group": str(value),
            "n": len(g),
            "positive": int(
                (g["label"] == 1).sum()
            ),
            "negative": int(
                (g["label"] == 0).sum()
            ),
            "ROC_AUC": roc_auc_score(
                g["label"],
                g["predicted_probability"]
            ),
            "PR_AUC": average_precision_score(
                g["label"],
                g["predicted_probability"]
            ),
        })

    return pd.DataFrame(rows)


subgroup_tables = []

for col in [
    "MHC_Class_Used",
    "Organism",
    "query_organism"
]:

    if col in df_eval.columns:

        temp = subgroup_auc_table(
            df_eval,
            col,
            min_n=10
        )

        subgroup_tables.append(
            temp
        )

subgroup_results = pd.concat(
    subgroup_tables,
    ignore_index=True
)

display(
    subgroup_results
)

subgroup_results.to_csv(
    outdir
    /
    "external_subgroup_performance.csv",
    index=False
)

#Cell 21 — Save final validation report

In [ ]:
report = {
    "model_file": MODEL_PATH,
    "external_dataset": CSV_PATH,

    "n_external": int(
        len(df)
    ),

    "n_positive": int(
        (y_true == 1).sum()
    ),

    "n_negative": int(
        (y_true == 0).sum()
    ),

    "training_feature_count":
        int(len(feature_columns)),

    "generated_feature_count":
        int(len(feature_only_cols)),

    "final_input_feature_count":
        int(X.shape[1]),

    "missing_training_features":
        missing_features,

    "extra_generated_features":
        extra_features,

    "esm_model":
        cfg.esm_model_name,

    "esm_enabled":
        bool(cfg.use_esm_embeddings),

    "decision_threshold":
        threshold,

    "metrics":
        metrics,

    "bootstrap_successful_replicates":
        n_success,

    "bootstrap_CI95":
        bootstrap_df.to_dict(
            orient="records"
        ),
}

with open(
    outdir
    /
    "external_validation_report.json",
    "w"
) as f:

    json.dump(
        report,
        f,
        indent=2,
        default=str
    )

with open(
    outdir
    /
    "external_validation_report.txt",
    "w"
) as f:

    f.write(
        "EXTERNAL VALIDATION REPORT\n"
    )

    f.write(
        "=" * 70 + "\n\n"
    )

    f.write(
        f"Model: {MODEL_PATH}\n"
    )

    f.write(
        f"Dataset: {CSV_PATH}\n"
    )

    f.write(
        f"N external: {len(df)}\n"
    )

    f.write(
        f"Positive: {(y_true == 1).sum()}\n"
    )

    f.write(
        f"Negative: {(y_true == 0).sum()}\n\n"
    )

    f.write(
        "PERFORMANCE METRICS\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    for key, value in metrics.items():

        f.write(
            f"{key}: {value}\n"
        )

    f.write(
        "\n\nBOOTSTRAP 95% CI\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    f.write(
        bootstrap_df.to_string(
            index=False
        )
    )

print(
    "\nExternal validation completed."
)

print(
    "\nAll results are in:"
)

print(
    outdir.resolve()
)

#Cell 22 — Download all results

In [ ]:
import shutil

zip_path = shutil.make_archive(
    "external_validation_results",
    "zip",
    root_dir=outdir
)

print(
    f"Created: {zip_path}"
)

from google.colab import files

files.download(
    zip_path
)